#  Notebook 01 — Exploratory Data Analysis

This notebook examines all three raw datasets before any modelling.

**What we cover:**
1. Load all 3 raw datasets
2. Shape, dtypes, memory, duplicates
3. ERA5 column audit + unit conversion check
4. NASA POWER column audit
5. Missing value analysis
6. Target variable deep-dive (PRECTOTCORR)
7. Feature distributions
8. Correlation heatmap
9. Outlier analysis
10. Temporal and spatial patterns

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy.stats import gaussian_kde
import warnings
from config_utils import load_config

warnings.filterwarnings('ignore')
plt.style.use('dark_background')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)
cfg = load_config(PROJECT_ROOT / 'configs' / 'config.yaml')
print(f'Libraries loaded from {PROJECT_ROOT} ✅')

## 1 Load Raw Datasets

In [ ]:
era5   = pd.read_csv(cfg['paths']['raw_era5'], parse_dates=['date'])
nasa   = pd.read_csv(cfg['paths']['raw_nasa'], parse_dates=['date'])
coords = pd.read_csv(cfg['paths']['raw_coords'])
print(f'ERA5   : {era5.shape}')
print(f'NASA   : {nasa.shape}')
print(f'Coords : {coords.shape}')

## 2️ ERA5 Dataset Audit

In [ ]:
print('=== ERA5 Columns ===')
print(era5.dtypes)
print('\nFirst 3 rows:')
era5.head(3)

In [ ]:
print('Missing values:')
print(era5.isnull().sum())
print(f'\nDuplicates: {era5.duplicated().sum()}')
print(f'Districts: {era5["district"].nunique()}')
print(f'Rows per district: {era5.groupby("district").size().describe()}')
print(f'Date range: {era5["date"].min()} → {era5["date"].max()}')

In [ ]:
# ERA5 unit conversion check
era5_c = era5.copy()
era5_c['t2m_c'] = era5_c['t2m'] - 273.15
era5_c['d2m_c'] = era5_c['d2m'] - 273.15
era5_c['sp_hpa'] = era5_c['sp'] / 100
era5_c['tp_mm'] = era5_c['tp'] * 1000
print('After unit conversion:')
for col, unit in [('t2m_c','°C'), ('d2m_c','°C'), ('sp_hpa','hPa'), ('tp_mm','mm/day')]:
    print(f'  {col}: {era5_c[col].min():.2f} → {era5_c[col].max():.2f} {unit}')

## 3️ NASA POWER Dataset Audit

In [ ]:
print('=== NASA Columns ===')
print(nasa.dtypes)
print('\nFirst 3 rows:')
nasa.head(3)

In [ ]:
print('Missing values:')
print(nasa.isnull().sum())
print(f'\nDistricts: {nasa["district"].nunique()}')
# Kanpur check
kanpur_districts = [d for d in nasa['district'].unique() if 'kanpur' in d.lower()]
print(f'\nKanpur entries: {kanpur_districts}')
print(f'Rows per entry: {nasa[nasa["district"].isin(kanpur_districts)].groupby("district").size().to_dict()}')

In [ ]:
# NASA precipitation outlier investigation
print('PRECTOTCORR stats:')
print(nasa['PRECTOTCORR'].describe())
print(f'\nValues > 100mm: {(nasa["PRECTOTCORR"] > 100).sum()}')
print(f'Values > 50mm: {(nasa["PRECTOTCORR"] > 50).sum()}')
# Find the 279mm event
big = nasa.nlargest(5, 'PRECTOTCORR')[['date','district','PRECTOTCORR']]
print(f'\nTop 5 precipitation events:')
print(big)

In [ ]:
# NASA rain day % at different thresholds
for thresh in [0, 1, 2.5, 5, 10]:
    pct = (nasa['PRECTOTCORR'] > thresh).mean() * 100
    print(f'  Days > {thresh:4.1f}mm: {pct:.1f}%')

## 4️ Coordinates Audit

In [ ]:
print(coords.dtypes)
coords.head(10)

In [ ]:
# Check name format mismatch
era5_dists = set(era5['district'].unique())
coords_norm = set(coords['district'].str.lower().str.replace(' ', '_'))
print(f'ERA5 districts: {len(era5_dists)}')
print(f'Coords normalized: {len(coords_norm)}')
print(f'Match: {len(era5_dists & coords_norm)}/72')

## 5️ Target Variable Deep-Dive

In [ ]:
# Which precipitation to use as target?
era5_c['month'] = era5_c['date'].dt.month
nasa['month'] = nasa['date'].dt.month
# Lucknow July comparison
lko_era5 = era5_c[(era5_c['district']=='lucknow') & (era5_c['month']==7)]
lko_nasa = nasa[(nasa['district']=='lucknow') & (nasa['month']==7)]
print('Lucknow July mean precipitation:')
print(f'  ERA5 tp (×1000 mm): {lko_era5["tp_mm"].mean():.2f} mm/day')
print(f'  NASA PRECTOTCORR:   {lko_nasa["PRECTOTCORR"].mean():.2f} mm/day')
print('\nConclusion: NASA PRECTOTCORR is the realistic target (IMD-validated)')

## 6️ Full EDA Plots — Run Visualize Module

In [ ]:
from visualize import run_all_eda_plots
run_all_eda_plots(cfg['_meta']['config_path'])
print(f"All EDA plots saved to {cfg['paths']['plots']}")